# Week 5 &mdash; Manifolds and Riemannian Geometry

## Implementation: The Cotangent Laplacian, Spectral Modes, and Heat Diffusion on a Mesh

We build a triangle mesh, assemble its **cotangent Laplace&ndash;Beltrami operator**, compute its **eigenfunctions** (the manifold Fourier basis), simulate **heat diffusion**, and derive an isometry-invariant **Heat Kernel Signature**. The implementation uses only NumPy/SciPy so every step is transparent.

### Setup


In [ ]:
# !pip install numpy scipy matplotlib

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

np.set_printoptions(precision=3, suppress=True)


## 1. Building a Mesh

We tessellate a surface of revolution &mdash; a **torus** &mdash; into triangles. A torus is a closed 2-manifold of genus 1, which makes its spectral and topological properties interesting.


In [ ]:
def torus_mesh(nu=40, nv=20, R=2.0, r=0.7):
    us = np.linspace(0, 2*np.pi, nu, endpoint=False)
    vs = np.linspace(0, 2*np.pi, nv, endpoint=False)
    U, V = np.meshgrid(us, vs, indexing="ij")
    X = (R + r*np.cos(V)) * np.cos(U)
    Y = (R + r*np.cos(V)) * np.sin(U)
    Z = r*np.sin(V)
    verts = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)

    def idx(i, j):
        return (i % nu) * nv + (j % nv)

    faces = []
    for i in range(nu):
        for j in range(nv):
            a, b, c, d = idx(i, j), idx(i+1, j), idx(i+1, j+1), idx(i, j+1)
            faces.append([a, b, c]); faces.append([a, c, d])
    return verts, np.array(faces)

V, F = torus_mesh()
print(f"Mesh: {len(V)} vertices, {len(F)} faces")


## 2. Assembling the Cotangent Laplacian

We implement the cotangent weights $\tfrac{1}{2}(\cot\alpha_{ij}+\cot\beta_{ij})$ and the lumped mass matrix $M$ of vertex (barycentric) areas, giving $L = M^{-1}C$.


In [ ]:
def cotangent_laplacian(V, F):
    n = len(V)
    C = sp.lil_matrix((n, n))
    mass = np.zeros(n)

    for tri in F:
        i, j, k = tri
        vi, vj, vk = V[i], V[j], V[k]
        # Edge vectors and the cotangent of the angle opposite each edge.
        e_ij = vj - vi; e_ik = vk - vi
        e_ji = vi - vj; e_jk = vk - vj
        e_ki = vi - vk; e_kj = vj - vk

        def cot(a, b):
            cross = np.linalg.norm(np.cross(a, b))
            return np.dot(a, b) / cross if cross > 1e-12 else 0.0

        # Angle at k is opposite edge (i,j); etc.
        cot_k = cot(e_ki, e_kj)   # opposite edge ij
        cot_i = cot(e_ij, e_ik)   # opposite edge jk
        cot_j = cot(e_ji, e_jk)   # opposite edge ik

        for (a, b, w) in [(i, j, cot_k), (j, k, cot_i), (i, k, cot_j)]:
            C[a, b] -= 0.5 * w
            C[b, a] -= 0.5 * w
            C[a, a] += 0.5 * w
            C[b, b] += 0.5 * w

        # Barycentric area: one third of triangle area to each vertex.
        area = 0.5 * np.linalg.norm(np.cross(vj - vi, vk - vi))
        mass[[i, j, k]] += area / 3.0

    M = sp.diags(mass)
    return C.tocsr(), M, mass

C, M, mass = cotangent_laplacian(V, F)
print("Stiffness matrix C: shape", C.shape, "| symmetric?",
      np.allclose((C - C.T).toarray(), 0, atol=1e-8))
print("Total surface area (sum of mass):", round(mass.sum(), 4))


## 3. The Manifold Fourier Basis

We solve the generalised eigenproblem $C\phi = \lambda M\phi$ for the lowest eigenpairs. The eigenfunctions $\phi_k$ are the *vibration modes* of the surface &mdash; the manifold analogue of sines and cosines.


In [ ]:
# Lowest k generalised eigenpairs (smallest |lambda|).
k = 12
evals, evecs = spla.eigsh(C, k=k, M=M, sigma=0, which="LM")
order = np.argsort(evals)
evals, evecs = evals[order], evecs[:, order]
print("Smallest Laplace-Beltrami eigenvalues:\n", np.round(evals, 4))
print("\nlambda_0 ~ 0 (constant mode)?", np.isclose(evals[0], 0, atol=1e-6))


In [ ]:
def plot_field_on_mesh(V, F, field, title, ax):
    polys = [V[f] for f in F]
    facecolor = plt.cm.coolwarm((field - field.min()) /
                                (np.ptp(field) + 1e-12))
    face_vals = field[F].mean(axis=1)
    norm = (face_vals - face_vals.min()) / (np.ptp(face_vals) + 1e-12)
    coll = Poly3DCollection(polys, facecolors=plt.cm.coolwarm(norm),
                            edgecolors="none", alpha=0.95)
    ax.add_collection3d(coll)
    ax.set_xlim(V[:,0].min(), V[:,0].max())
    ax.set_ylim(V[:,1].min(), V[:,1].max())
    ax.set_zlim(V[:,2].min(), V[:,2].max())
    ax.set_box_aspect((np.ptp(V[:,0]), np.ptp(V[:,1]), np.ptp(V[:,2])))
    ax.set_title(title); ax.axis("off")

fig = plt.figure(figsize=(12, 3.4))
for n, m in enumerate([1, 2, 5, 9]):
    ax = fig.add_subplot(1, 4, n+1, projection="3d")
    plot_field_on_mesh(V, F, evecs[:, m], f"$\\phi_{{{m}}}$", ax)
plt.suptitle("Laplace-Beltrami eigenfunctions (manifold Fourier modes)")
plt.tight_layout(); plt.show()


Low-index modes vary slowly across the surface; higher modes oscillate more rapidly &mdash; exactly the frequency interpretation from the theory notebook, now on a curved domain.


## 4. Heat Diffusion on the Mesh

We simulate the heat equation $\partial_t u = \Delta_{\mathcal M} u$ via the spectral solution $u(t) = \sum_k e^{-\lambda_k t}\,\langle u_0, \phi_k\rangle_M\,\phi_k$, starting from a delta-like spike at one vertex.


In [ ]:
# Use a larger eigenbasis for an accurate heat kernel.
kk = 80
ev, ef = spla.eigsh(C, k=kk, M=M, sigma=0, which="LM")
o = np.argsort(ev); ev, ef = ev[o], ef[:, o]

src = 0
u0 = np.zeros(len(V)); u0[src] = 1.0
coeffs = ef.T @ (M @ u0)          # M-inner-product projection

def heat(t):
    return ef @ (np.exp(-ev * t) * coeffs)

fig = plt.figure(figsize=(12, 3.4))
for n, t in enumerate([0.02, 0.1, 0.4, 1.5]):
    ax = fig.add_subplot(1, 4, n+1, projection="3d")
    plot_field_on_mesh(V, F, heat(t), f"$t = {t}$", ax)
plt.suptitle("Heat diffusion from a point source on the torus")
plt.tight_layout(); plt.show()


Heat spreads outward along the surface, following its intrinsic geometry &mdash; it travels *around* the torus, not through the hole. The rate and pattern of diffusion encode the surface's shape.


## 5. An Isometry-Invariant Descriptor: the Heat Kernel Signature

The **HKS** at vertex $i$ records the amount of heat retained at $i$ over time:
$$
\mathrm{HKS}_i(t) = \sum_{k} e^{-\lambda_k t}\,\phi_k(i)^2.
$$
Because it is built entirely from the Laplace&ndash;Beltrami spectrum, it is **invariant to isometric deformations** of the surface &mdash; the property that makes it valuable for non-rigid shape matching and segmentation (Week 6).


In [ ]:
ts = np.logspace(-2, 0.5, 16)
HKS = np.array([[ (np.exp(-ev * t) * ef[i]**2).sum() for t in ts ]
                for i in range(len(V)) ])
print("HKS feature matrix shape (vertices x time-scales):", HKS.shape)

# Colour the mesh by HKS at one scale: geometrically distinctive regions stand out.
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(111, projection="3d")
plot_field_on_mesh(V, F, HKS[:, 6], "HKS at one time-scale", ax)
plt.tight_layout(); plt.show()


## 6. Summary and Exercises

- We assembled the **cotangent Laplacian** $L = M^{-1}C$ &mdash; the discrete Laplace&ndash;Beltrami operator on a mesh.
- We computed its **eigenfunctions** (the manifold Fourier basis) and visualised low- vs. high-frequency modes.
- We simulated **heat diffusion** spectrally and observed it respect the surface's intrinsic geometry.
- We derived the **Heat Kernel Signature**, an isometry-invariant per-vertex descriptor.

These spectral and diffusion tools are exactly the per-vertex features and operators we feed into a GNN for the **shape-segmentation final project** in Week 6.

### Exercises

1. Verify numerically that the HKS is unchanged when the mesh is rigidly rotated/translated. (It should be, since it is intrinsic.)
2. Reconstruct a smooth function on the mesh from only its first $k$ Laplace&ndash;Beltrami coefficients and study the error as $k$ grows (spectral compression).
3. Implement implicit-Euler heat diffusion $(M - tC)\,u_{n+1} = M u_n$ and compare with the spectral solution for large $t$.
